# Piloto OmniSpatial — InternVL2 2B

**Benchmark:** OmniSpatial (split test, 1 533 preguntas, opción múltiple A/B/C/D)  
**Modelo:** OpenGVLab/InternVL2-2B — `trust_remote_code=True`, interfaz `.chat()`  
**Tracks evaluados:** Geometric\_Reasoning · Motion\_Analysis · Egocentric · Allocentric  

**Diferencias vs. otros modelos:**
- Usa `AutoModel` + `AutoTokenizer` (no `AutoModelForCausalLM` ni `PaliGemmaForConditionalGeneration`).
- Preproceso de imagen manual con `torchvision.transforms` (sin `processor` unificado).
- Inferencia vía método `.chat()` propio del modelo.

In [1]:
# einops y timm son requeridos por el codigo remoto de InternVL2
!pip install -q "transformers==4.37.2" accelerate einops timm pillow psutil huggingface_hub "sentencepiece==0.1.99"

# Limpiamos la caché específica de este modelo por si el tokenizer.model está corrupto
!rm -rf ~/.cache/huggingface/hub/models--OpenGVLab--InternVL2-2B

print('Dependencias instaladas y caché limpiada.')

  Preparing metadata (setup.py) ... done
Dependencias instaladas y caché limpiada.


In [2]:
import transformers, einops, timm
print(f'transformers: {transformers.__version__}')
print(f'einops:       {einops.__version__}')
print(f'timm:         {timm.__version__}')
print('Entorno OK.')

transformers: 4.37.2
einops:       0.8.2
timm:         1.0.28
Entorno OK.


In [3]:
MODEL_ID            = 'OpenGVLab/InternVL2-2B'
BENCHMARK_NAME      = 'OmniSpatial'
IMAGE_SIZE          = 448  # resolucion de entrada del modelo

TRACKS              = ['Geometric_Reasoning', 'Motion_Analysis', 'Egocentric', 'Allocentric']
N_SAMPLE_PER_TRACK  = 100
RANDOM_SEED         = 42
TIMEOUT_SEGUNDOS    = 60

modelo_short = MODEL_ID.split('/')[-1]
OUTPUT_TXT   = f'/content/resultados_{BENCHMARK_NAME}_{modelo_short}.txt'
OUTPUT_CSV   = f'/content/detalle_{BENCHMARK_NAME}_{modelo_short}.csv'

print(f'Modelo:   {MODEL_ID}')
print(f'Tracks:   {TRACKS}')
print(f'Muestra:  {N_SAMPLE_PER_TRACK}/track = {N_SAMPLE_PER_TRACK * len(TRACKS)} total')
print(f'Seed:     {RANDOM_SEED}')
print(f'Salida:   {OUTPUT_TXT}')

Modelo:   OpenGVLab/InternVL2-2B
Tracks:   ['Geometric_Reasoning', 'Motion_Analysis', 'Egocentric', 'Allocentric']
Muestra:  100/track = 400 total
Seed:     42
Salida:   /content/resultados_OmniSpatial_InternVL2-2B.txt


In [4]:
import torch
import torchvision.transforms as T

# Normalizacion estandar de InternVL2 (misma que ImageNet)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

img_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE),
              interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def preparar_imagen(imagen_pil):
    """Convierte PIL.Image -> tensor [1, 3, H, W] en bfloat16 en el device."""
    return (
        img_transform(imagen_pil.convert('RGB'))
        .unsqueeze(0)
        .to(torch.bfloat16)
        .to(device)
    )


print(f'Transform definido para {IMAGE_SIZE}x{IMAGE_SIZE}.')

Transform definido para 448x448.


In [5]:
from transformers import AutoModel, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cpu':
    print('ADVERTENCIA: sin GPU. Activar T4 en Entorno de ejecucion > Cambiar tipo de entorno.')

if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map='auto',
).eval()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,  # Agregado para evitar el error de caracteres nulos en sentencepiece
)

vram_carga_mb = torch.cuda.max_memory_allocated() / 1024**2 if device == 'cuda' else 0
print(f'Modelo cargado. VRAM tras la carga: {vram_carga_mb:.1f} MB')

GEN_CONFIG = dict(max_new_tokens=64, do_sample=False)

# ── Interfaz comun ───────────────────────────────────────────────────────────
def predecir(imagen_pil, prompt_texto):
    """Recibe PIL.Image y el prompt completo. Devuelve string crudo del modelo.

    InternVL2: preproceso de imagen manual, luego model.chat() con <image>\n
    Como prefijo para que el modelo ubique el token de imagen.
    """
    pixel_values = preparar_imagen(imagen_pil)
    question     = f'<image>\n{prompt_texto}'
    return model.chat(tokenizer, pixel_values, question, GEN_CONFIG)
# ─────────────────────────────────────────────────────────────────────────────

print('Interfaz predecir() lista.')

Dispositivo: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_internvl_chat.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL2-2B:
- configuration_internvl_chat.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_internvl_chat.py: 0.00B [00:00, ?B/s]

... (8 outputs left out)

added_tokens.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Modelo cargado. VRAM tras la carga: 4592.9 MB
Interfaz predecir() lista.


In [6]:
print('cuda disponible:  ', torch.cuda.is_available())
print('device del modelo:', next(model.parameters()).device)
if torch.cuda.is_available():
    print('GPU:              ', torch.cuda.get_device_name(0))
import subprocess
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv'],
    capture_output=True, text=True
)
print(r.stdout or 'nvidia-smi no disponible')

cuda disponible:   True
device del modelo: cuda:0
GPU:               Tesla T4
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 4875 MiB, 15360 MiB



In [7]:
import os, subprocess
from huggingface_hub import snapshot_download

BASE_DIR  = '/content/dataset/OmniSpatial-test'
JSON_PATH = os.path.join(BASE_DIR, 'data.json')

if os.path.exists(JSON_PATH):
    print('data.json ya existe, se omite la descarga.')
else:
    print('Descargando OmniSpatial-test.zip (~1.6 GB)...')
    snapshot_download(
        repo_id='qizekun/OmniSpatial',
        repo_type='dataset',
        local_dir='/content/dataset',
        allow_patterns=['OmniSpatial-test.zip'],
    )
    print('Descomprimiendo...')
    subprocess.run(['unzip', '-q', '-o', 'OmniSpatial-test.zip'],
                   cwd='/content/dataset', check=True)
    print('Listo.')

r = subprocess.run(['ls', '-lh', '/content/dataset/'], capture_output=True, text=True)
print(r.stdout)

Descargando OmniSpatial-test.zip (~1.6 GB)...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

OmniSpatial-test.zip:   0%|          | 0.00/1.66G [00:00<?, ?B/s]

Descomprimiendo...
Listo.
total 1.6G
drwxr-xr-x 3 root root 4.0K Jul 26 03:13 __MACOSX
drwxr-xr-x 6 root root 4.0K Aug 11  2025 OmniSpatial-test
-rw-r--r-- 1 root root 1.6G Jul 26 03:13 OmniSpatial-test.zip



In [8]:
import json, os
import pandas as pd

with open(JSON_PATH, encoding='utf-8') as f:
    data = json.load(f)

df_full = pd.DataFrame(data)
df_full['row_idx'] = range(len(df_full))

media_files = {}
for root, _, files in os.walk(BASE_DIR):
    for fname in files:
        ext = os.path.splitext(fname)[1].lower()
        if ext in ('.jpg', '.jpeg', '.png'):
            stem = os.path.splitext(fname)[0]
            media_files[stem] = os.path.join(root, fname)

df_full['media_id']   = df_full['id'].apply(lambda x: x.split('_')[0])
df_full['media_path'] = df_full['media_id'].map(media_files)
df_full['has_opts']   = df_full['options'].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

print(f'Total preguntas   : {len(df_full)}')
print(f'Con imagen mapeada: {df_full["media_path"].notna().sum()} / {len(df_full)}')
print()
print('Distribucion por sub_task_type:')
display(df_full['sub_task_type'].value_counts().to_frame('n'))

Total preguntas   : 1533
Con imagen mapeada: 1533 / 1533

Distribucion por sub_task_type:


,n
sub_task_type,
Allocentric,376
Motion_Analysis,346
Geometric_Reasoning,155
Geospatial_Strategy,110
Localization,105
Egocentric,102
Pattern_Recognition,97
Traffic_Analysis,85
Hypothetical,83


In [9]:
partes = []
print('Muestreo por track:')

for track in TRACKS:
    pool = df_full[
        (df_full['sub_task_type'] == track) &
        df_full['media_path'].notna()
    ].copy()

    n = min(N_SAMPLE_PER_TRACK, len(pool))

    if track == 'Geometric_Reasoning':
        con = pool[pool['has_opts']]
        sin = pool[~pool['has_opts']]
        n_con = round(n * len(con) / len(pool))
        n_sin = n - n_con
        muestra = pd.concat([
            con.sample(min(n_con, len(con)), random_state=RANDOM_SEED),
            sin.sample(min(n_sin, len(sin)), random_state=RANDOM_SEED),
        ])
        print(f'  {track:25s}: {len(muestra)} '
              f'({len(con)} con opts, {len(sin)} sin opts en pool)')
    else:
        muestra = pool.sample(n, random_state=RANDOM_SEED)
        print(f'  {track:25s}: {len(muestra)} de {len(pool)}')

    partes.append(muestra)

df_muestra = (
    pd.concat(partes)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)
print(f'\nTotal muestra: {len(df_muestra)}')
print(df_muestra['sub_task_type'].value_counts().to_string())

Muestreo por track:
  Geometric_Reasoning      : 100 (22 con opts, 133 sin opts en pool)
  Motion_Analysis          : 100 de 346
  Egocentric               : 100 de 102
  Allocentric              : 100 de 376

Total muestra: 400
sub_task_type
Egocentric             100
Geometric_Reasoning    100
Allocentric            100
Motion_Analysis        100


In [10]:
import re

LETRAS = ['A', 'B', 'C', 'D']


def construir_prompt(row):
    q    = row['question']
    opts = row['options'] if isinstance(row['options'], list) else []
    if opts:
        lineas = '\n'.join(f'{LETRAS[i]}. {o}' for i, o in enumerate(opts))
        return (
            f'Question: {q}\n'
            f'Options:\n{lineas}\n'
            'Answer with a single letter: A, B, C, or D.'
        )
    else:
        return (
            f'Question: {q}\n'
            'The answer options are labeled A, B, C, and D '
            'and shown as figures in the image.\n'
            'Answer with a single letter: A, B, C, or D.'
        )


def parsear_respuesta(texto):
    if not texto:
        return None
    t = texto.upper().strip()

    m = re.match(r'^([ABCD])[.\s,]', t)
    if m:
        return LETRAS.index(m.group(1))

    m = re.search(r'\b([ABCD])\b', t)
    if m:
        return LETRAS.index(m.group(1))

    m = re.search(r'(?:ANSWER|OPTION|CHOICE|LETTER)\s+(?:IS\s+)?([ABCD])', t)
    if m:
        return LETRAS.index(m.group(1))

    return None


assert parsear_respuesta('A')                   == 0
assert parsear_respuesta('B. to the right')     == 1
assert parsear_respuesta('The answer is C')     == 2
assert parsear_respuesta('Option D is correct') == 3
assert parsear_respuesta('I cannot determine')  is None
print('construir_prompt() y parsear_respuesta() OK.')

construir_prompt() y parsear_respuesta() OK.


In [11]:
import time, concurrent.futures
from PIL import Image


def query_con_timeout(imagen, prompt, timeout_seg):
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    future   = executor.submit(predecir, imagen, prompt)
    try:
        resultado = future.result(timeout=timeout_seg)
        executor.shutdown(wait=False)
        return resultado, False
    except concurrent.futures.TimeoutError:
        executor.shutdown(wait=False)
        return None, True


resultados   = []
n_err_imagen = 0

print(f'Iniciando: {len(df_muestra)} preguntas (timeout={TIMEOUT_SEGUNDOS}s)...\n')

for fila_i, row in df_muestra.iterrows():
    try:
        img = Image.open(row['media_path']).convert('RGB')
    except Exception as e:
        n_err_imagen += 1
        print(f'[ERR imagen] {row["media_path"]}: {e}')
        continue

    prompt = construir_prompt(row)

    t0 = time.time()
    respuesta_cruda, timed_out = query_con_timeout(img, prompt, TIMEOUT_SEGUNDOS)
    t1 = time.time()

    if timed_out:
        pred_idx   = None
        parse_fail = False
    else:
        pred_idx   = parsear_respuesta(respuesta_cruda)
        parse_fail = (pred_idx is None)

    acierto = (pred_idx == row['answer']) if pred_idx is not None else False

    resultados.append({
        'row_idx':         row['row_idx'],
        'id':              row['id'],
        'track':           row['sub_task_type'],
        'has_opts':        row['has_opts'],
        'question':        row['question'],
        'ground_truth':    row['answer'],
        'respuesta_cruda': respuesta_cruda,
        'prediccion':      pred_idx,
        'pred_letra':      LETRAS[pred_idx] if pred_idx is not None else None,
        'parse_fail':      parse_fail,
        'timeout':         timed_out,
        'acierto':         acierto,
        'tiempo_seg':      t1 - t0,
    })

    n = len(resultados)
    if timed_out:
        print(f'[{n}/{len(df_muestra)}] TIMEOUT [{row["sub_task_type"]}]')
    elif n == 1 or n % 50 == 0:
        gt_l = LETRAS[row['answer']]
        pl   = LETRAS[pred_idx] if pred_idx is not None else '?'
        print(f'[{n}/{len(df_muestra)}] GT={gt_l} Pred={pl} '
              f'"{str(respuesta_cruda)[:60]}" {t1-t0:.1f}s [{row["sub_task_type"]}]')

print(f'\nFin. Procesadas={len(resultados)}, err_imagen={n_err_imagen}')

Iniciando: 400 preguntas (timeout=60s)...

[1/400] GT=A Pred=C "C" 3.9s [Egocentric]
[50/400] GT=C Pred=C "C" 0.9s [Motion_Analysis]
[100/400] GT=B Pred=A "A, B, C, or D." 1.2s [Geometric_Reasoning]
[150/400] GT=D Pred=C "C" 0.9s [Allocentric]


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[200/400] GT=A Pred=A "A" 0.9s [Allocentric]
[250/400] GT=D Pred=D "D" 0.9s [Egocentric]
[300/400] GT=A Pred=A "A" 0.9s [Egocentric]
[350/400] GT=A Pred=A "A" 0.9s [Geometric_Reasoning]
[400/400] GT=B Pred=B "B" 0.9s [Motion_Analysis]

Fin. Procesadas=400, err_imagen=0


In [12]:
import psutil

df_res = pd.DataFrame(resultados)

vram_peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if device == 'cuda' else 0
ram_mb       = psutil.Process().memory_info().rss / 1024**2

n_total      = len(df_res)
n_timeout    = int(df_res['timeout'].sum())
n_parse_fail = int(df_res['parse_fail'].sum())
timeout_rate    = n_timeout    / n_total if n_total else float('nan')
parse_fail_rate = n_parse_fail / n_total if n_total else float('nan')

df_val     = df_res[~df_res['timeout'] & ~df_res['parse_fail']]
acc_global = df_val['acierto'].mean()

t_no_timeout = df_res[~df_res['timeout']]
t_medio      = t_no_timeout['tiempo_seg'].mean()
t_std        = t_no_timeout['tiempo_seg'].std()

LINEA = '=' * 65
print(LINEA)
print(f'RESULTADOS  {BENCHMARK_NAME}  {MODEL_ID}')
print(LINEA)
print(f'Preguntas totales  : {n_total}')
print(f'Timeouts           : {n_timeout} ({timeout_rate:.1%})')
print(f'Parse fails        : {n_parse_fail} ({parse_fail_rate:.1%})')
print(f'Accuracy global    : {acc_global:.1%}  (n={len(df_val)})')
print(f'Tiempo medio       : {t_medio:.2f}s +/- {t_std:.2f}s')
print(f'VRAM pico          : {vram_peak_mb:.1f} MB')
print(f'RAM pico           : {ram_mb:.1f} MB')

print('\n--- Accuracy por track ---')
for track in TRACKS:
    g   = df_val[df_val['track'] == track]
    tot = len(df_res[df_res['track'] == track])
    if len(g) == 0:
        print(f'  {track:25s}: sin datos validos')
        continue
    baseline = (
        df_res[df_res['track'] == track]['ground_truth']
        .value_counts(normalize=True).max()
    )
    print(f'  {track:25s}: acc={g["acierto"].mean():.1%}  '
          f'n_val={len(g)}/{tot}  baseline={baseline:.1%}')

g_geo = df_val[df_val['track'] == 'Geometric_Reasoning']
if len(g_geo) > 0:
    print('\n--- Geometric_Reasoning: con opciones vs. sin opciones ---')
    for tiene_opts, sub in g_geo.groupby('has_opts'):
        etq = 'con opciones texto' if tiene_opts else 'sin opciones (figuras en imagen)'
        print(f'  {etq:35s}: acc={sub["acierto"].mean():.1%}  n={len(sub)}')

print('\n--- Distribucion de predicciones (detectar colapso) ---')
print(df_val['pred_letra'].value_counts().to_string())

RESULTADOS  OmniSpatial  OpenGVLab/InternVL2-2B
Preguntas totales  : 400
Timeouts           : 0 (0.0%)
Parse fails        : 0 (0.0%)
Accuracy global    : 39.2%  (n=400)
Tiempo medio       : 0.96s +/- 0.17s
VRAM pico          : 4969.3 MB
RAM pico           : 2519.8 MB

--- Accuracy por track ---
  Geometric_Reasoning      : acc=27.0%  n_val=100/100  baseline=27.0%
  Motion_Analysis          : acc=42.0%  n_val=100/100  baseline=31.0%
  Egocentric               : acc=60.0%  n_val=100/100  baseline=36.0%
  Allocentric              : acc=28.0%  n_val=100/100  baseline=28.0%

--- Geometric_Reasoning: con opciones vs. sin opciones ---
  sin opciones (figuras en imagen)   : acc=24.4%  n=86
  con opciones texto                 : acc=42.9%  n=14

--- Distribucion de predicciones (detectar colapso) ---
pred_letra
A    194
C     80
D     67
B     59


In [13]:
import datetime

lineas = [
    f'BENCHMARK: {BENCHMARK_NAME}',
    f'MODELO: {MODEL_ID}',
    f'FECHA: {datetime.datetime.now().isoformat()}',
    f'TRACKS: {TRACKS}',
    f'N_SAMPLE_PER_TRACK: {N_SAMPLE_PER_TRACK}',
    f'SEMILLA: {RANDOM_SEED}',
    '-' * 50,
    f'TIMEOUT_RATE: {timeout_rate:.4f}',
    f'PARSE_FAIL_RATE: {parse_fail_rate:.4f}',
    f'ACCURACY_GLOBAL: {acc_global:.4f}',
    f'TIEMPO_MEDIO_SEG: {t_medio:.4f}',
    f'TIEMPO_STD_SEG: {t_std:.4f}',
    f'VRAM_PICO_MB: {vram_peak_mb:.2f}',
    f'RAM_PICO_MB: {ram_mb:.2f}',
    '-' * 50,
    'DESGLOSE_POR_TRACK',
]
for track in TRACKS:
    g   = df_val[df_val['track'] == track]
    tot = len(df_res[df_res['track'] == track])
    baseline = (
        df_res[df_res['track'] == track]['ground_truth']
        .value_counts(normalize=True).max()
        if tot > 0 else float('nan')
    )
    acc = g['acierto'].mean() if len(g) > 0 else float('nan')
    lineas.append(
        f'  {track}: n_val={len(g)} n_total={tot} '
        f'accuracy={acc:.4f} baseline_azar={baseline:.4f}'
    )

lineas += ['-' * 50, 'GEOMETRIC_REASONING_DETALLE']
g_geo = df_val[df_val['track'] == 'Geometric_Reasoning']
for tiene_opts, sub in g_geo.groupby('has_opts'):
    etq = 'con_opts' if tiene_opts else 'sin_opts'
    lineas.append(f'  {etq}: n={len(sub)} accuracy={sub["acierto"].mean():.4f}')

with open(OUTPUT_TXT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lineas) + '\n')
print(f'Guardado: {OUTPUT_TXT}')

df_res.to_csv(OUTPUT_CSV, index=False)
print(f'Guardado: {OUTPUT_CSV}')

try:
    from google.colab import files
    files.download(OUTPUT_TXT)
    files.download(OUTPUT_CSV)
except ImportError:
    print('(No es Colab) Descarga manual desde el panel de archivos.')

Guardado: /content/resultados_OmniSpatial_InternVL2-2B.txt
Guardado: /content/detalle_OmniSpatial_InternVL2-2B.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>